<a href="https://colab.research.google.com/github/Shamsfathalla/FlyRank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shamsfathalla/FlyRank-Starter-Notebooks/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: The paper claims that refreshing mature pages results in a 3.2x health boost.

My Question: Where does the "days since update" label come from? If this is pulled from a CMS timestamp, a simple backend plugin update could trigger it without any real editorial changes being made.

Finding 2: The paper's ML appendix shows a logistic regression model that predicts growth with 71% holdout accuracy.

My Question: Does the 80/20 holdout split group the data by page or client? If it was just a random row split, the model might have trained and tested on the exact same pages from different days, causing data leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I am moving from a random split to a grouped split based on client_hash_id. A random split lets the model memorize pages it has already seen on different dates. Grouping by client ensures my model is tested on entirely new data, giving a more realistic AUC score.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
table_path = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

q = f"""
    SELECT client_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, scroll_events
    FROM {table_path}
    WHERE month = '2026-03' AND gsc_impressions > 50
    LIMIT 10000
"""
df = con.sql(q).df()
df['is_opportunity'] = (df['gsc_clicks'] == 0).astype(int)

features = ['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'scroll_events']
X = df[features]
y = df['is_opportunity']

X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(X, y, test_size=0.2, random_state=42)
rf_naive = RandomForestClassifier(max_depth=3, random_state=42).fit(X_train_n, y_train_n)
naive_auc = roc_auc_score(y_test_n, rf_naive.predict_proba(X_test_n)[:, 1])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_hash_id']))
rf_honest = RandomForestClassifier(max_depth=3, random_state=42).fit(X.iloc[train_idx], y.iloc[train_idx])
honest_auc = roc_auc_score(y.iloc[test_idx], rf_honest.predict_proba(X.iloc[test_idx])[:, 1])

print(f"Before (Naive Split AUC): {naive_auc:.4f}")
print(f"After (Client Grouped Split AUC): {honest_auc:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Before (Naive Split AUC): 0.8040
After (Client Grouped Split AUC): 0.7376


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I am verifying that gsc_clicks (my target proxy) is completely removed from my training features. I am also running a correlation check to ensure no other feature is acting as a hidden shortcut to the answers.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Ensure clicks are excluded
assert 'gsc_clicks' not in features, "Leakage: Clicks found in features!"
assert 'is_opportunity' not in features, "Leakage: Target found in features!"

# Find suspiciously high correlations
correlations = X.corrwith(y).abs()
print("Feature correlations with target:")
print(correlations.round(3))

assert correlations.max() < 0.9, "Leakage: Suspiciously high correlation found!"
print("\nLeakage audit passed.")

Feature correlations with target:
gsc_impressions         0.292
gsc_avg_position        0.135
ga4_sessions            0.358
ga4_engaged_sessions    0.242
scroll_events           0.266
dtype: float64

Leakage audit passed.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Old unsafe claim: "My model predicts exactly which pages are failing and guarantees an increase in clicks if refreshed."

New safe claim: "My model provides directional decision-support by scoring pages based on observed search signals, helping teams prioritize which pages to review."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.